# Historic Ethereum evidence: Aave and Compound
This notebook needs an authenticated Ethereum archive RPC supporting `eth_getLogs` and historic `eth_call`. Add Colab secret **ETH_ARCHIVE_RPC**, and optionally an independent **ETH_VALIDATION_RPC**. Never print or commit credentials. This notebook does not itself prove full Compound or market-wide Health Factor coverage.

In [ ]:
from google.colab import userdata
from pathlib import Path
import subprocess,sys,os,json
ROOT=Path('/content/defi-systemic-risk')
if not ROOT.exists():
    subprocess.run(['git','clone','https://github.com/nikorokni/defi-systemic-risk.git',str(ROOT)],check=True)
os.chdir(ROOT)
subprocess.run([sys.executable,'-m','pip','install','-q','eth-hash[pycryptodome]','pandas','pyarrow'],check=True)
rpc=userdata.get('ETH_ARCHIVE_RPC')
if not rpc: raise RuntimeError('Enable secret ETH_ARCHIVE_RPC')
os.environ['ETH_ARCHIVE_RPC']=rpc
try:
    second=userdata.get('ETH_VALIDATION_RPC')
    if second: os.environ['ETH_VALIDATION_RPC']=second
except Exception: pass
print('RPC configured without printing secrets')

## Compound III USDC targeted log checks
Start with the missing August 2024 crisis neighborhood. This retrieves a single Comet USDC contract, **not** every Compound deployment or Compound V2 liquidation.

In [ ]:
out=Path('revision/results/chain_validation');out.mkdir(parents=True,exist_ok=True)
subprocess.run([sys.executable,'revision/scripts/chain_block_range.py','--start-utc','2024-08-01','--end-utc','2024-08-09','--out',str(out/'august_2024_bounds.json')],check=True)
bounds=json.loads((out/'august_2024_bounds.json').read_text())
subprocess.run([sys.executable,'revision/scripts/historical_validation.py','compound','--start-block',str(bounds['first_inclusive_block']),'--stop-block',str(bounds['last_inclusive_block']),'--out',str(out)],check=True)
print('Saved Compound III USDC log results; now compare to publisher series and a second independent source')

## Targeted Aave historical account mechanics
Ten selected 2023 accounts support spot checks only. They do not represent a complete borrower population and cannot validate an HF-based forecast. The validator aborts when historic account states cannot be reconstructed safely.

In [ ]:
subprocess.run([sys.executable,'revision/scripts/historical_validation.py','health','--accounts','revision/data/hf_spotcheck_requests.csv','--out',str(out)],check=True)
print('Saved selected historical HF checks; inspect discrepancies and unsupported cases')

## Export evidence, without secrets
If any command fails, retain the explicit error and do not replace missing responses with zero outcomes.

In [ ]:
import zipfile
package=Path('/content/Aave_Independent_Chain_Evidence.zip')
with zipfile.ZipFile(package,'w',zipfile.ZIP_DEFLATED) as z:
    for f in out.rglob('*'):
        if f.is_file():z.write(f,f.relative_to(ROOT))
from google.colab import files
files.download(str(package))